# Evaluate BLEU Score

In [1]:
import os
import torch
import evaluate
from datasets import load_dataset
from tqdm import tqdm
 
from configs.english_german_config import English_german_config
from model.Transformer import Transformer
from model.beam_search import BeamSearch
from model.bpe_tokenizer import build_and_train_BPE_tokenizer
from model.utils import load_checkpoint
from inference import translate_sentence

/Users/tonyavis/miniconda3/envs/transformer_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [3]:
def load_chpt(cfg:English_german_config, device)->Transformer:
    model = Transformer(cfg)
    chpt_path = os.path.join(cfg.MODEL_DIR, "checkpoints", cfg.checkpoint_name)

    if not os.path.exists(chpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {chpt_path}")
    
    print(f"Loading checkpoint from {chpt_path}...")
    checkpoint = torch.load(chpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model

In [ ]:
cfg = English_german_config()

# TODO: Update with better model
# NOTE: Config must be the same as the one used to train the this checkpoint!
cfg.checkpoint_name = "t"


In [5]:
tokenizer = build_and_train_BPE_tokenizer(
    cfg=cfg, perc_to_download=cfg.perc_to_download, dataset_iterator=None
)

model = load_chpt(cfg, device)


Loading existing BPE tokenizer from: (/Users/tonyavis/Main/AI_projects_and_res/Transformer/model/saved_models/tokenizer/wmt_14_shared_bpe_tokenizer_1_ds_percent.json)...
Loading checkpoint from /Users/tonyavis/Main/AI_projects_and_res/Transformer/model/checkpoints/transformer_epoch_1_1_percent_ds.pt...


In [6]:
# Load the BLEU metric
sacreblue = evaluate.load("sacrebleu")

Using the latest cached version of the module from /Users/tonyavis/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--sacrebleu/28676bf65b4f88b276df566e48e603732d0b4afd237603ebdf92acaacf5be99b (last modified on Fri Mar 20 07:38:01 2026) since it couldn't be found locally at evaluate-metric--sacrebleu, or remotely on the Hugging Face Hub.


In [7]:
print("Loading validation split of the dataset...")
val_ds = load_dataset("wmt14", "de-en", split="validation")

'[Errno 8] nodename nor servname provided, or not known' thrown while requesting HEAD https://huggingface.co/datasets/wmt14/resolve/b199e406369ec1b7634206d3ded5ba45de2fe696/wmt14.py
Retrying in 1s [Retry 1/5].


Loading validation split of the dataset...


Using the latest cached version of the dataset since wmt14 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'de-en' at /Users/tonyavis/.cache/huggingface/datasets/wmt14/de-en/0.0.0/b199e406369ec1b7634206d3ded5ba45de2fe696 (last modified on Fri Mar 20 07:40:24 2026).


In [8]:
# Test on a subset to save time
num_samples = 100
subset = val_ds.select(range(num_samples))

predictions = []
references = []

In [9]:
print(f"Translating {num_samples} sentences for evaluation...")
for i in tqdm(subset):
    eng_text = i["translation"]["en"]
    target_de_text = i["translation"]["de"]

    pred_text = translate_sentence(eng_text, model, tokenizer, cfg, device)

    predictions.append(pred_text)
    references.append([target_de_text])

Translating 100 sentences for evaluation...


100%|██████████| 100/100 [02:44<00:00,  1.65s/it]


In [10]:
print("\nCalculating BLEU score...")
results = sacreblue.compute(predictions=predictions, references=references)
print(f"\n Final BLEU score: {results['score']:.2f}")


Calculating BLEU score...

 Final BLEU score: 0.05
